# Direct ST: Literal `<unk>` Blocking Sensitivity Test

This notebook tests whether preventing SeamlessM4T from generating the **ordinary token sequence that literally spells `<unk>`** improves the Direct translation output.

This is **not fine-tuning**. Model weights stay frozen. The only intervention is a decoding constraint using `bad_words_ids`.

The notebook compares:

1. **Baseline greedy** — original Direct decoding.
2. **Greedy + block literal `<unk>`** — same settings, but the discovered ordinary-token sequences that decode to `<unk>` are forbidden.

The original 4,200 Direct predictions are never overwritten. This is a post-hoc sensitivity experiment.


## 1. Install dependencies

For the cleanest final comparison, run on the same **Tesla T4 + FP16** environment used for the original Direct run.


In [1]:
%pip install -q "transformers==4.57.6" sentencepiece protobuf pandas numpy matplotlib sacrebleu huggingface_hub librosa soundfile


You should consider upgrading via the '/Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
from datetime import datetime, timezone
import json
import random
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sacrebleu
import torch
import torchaudio
import transformers

from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToText

print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Transformers:", transformers.__version__)
print("SacreBLEU:", sacrebleu.__version__)
print("CUDA available:", torch.cuda.is_available())


/Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


PyTorch: 2.8.0
Torchaudio: 2.8.0
Transformers: 4.57.6
SacreBLEU: 2.6.0
CUDA available: False


## 2. Configuration

Start with a small balanced test. After that works, switch `TEST_SCOPE` to `"all_affected"`.


In [3]:
SEED = 760

MODEL_ID = "facebook/seamless-m4t-v2-large"
MODEL_REVISION = "5f8cc790b19fc3f67a61c105133b20b34e3dcb76"
TARGET_LANGUAGE = "cmn"

TEST_SCOPE = "balanced_small"   # "balanced_small" or "all_affected"
SAMPLES_PER_ACCENT = 5

GENERATION_KWARGS = {
    "num_beams": 1,
    "do_sample": False,
    "max_new_tokens": 256,
}

MAX_LITERAL_UNK_NGRAM = 6

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Generation settings:", GENERATION_KWARGS)


Generation settings: {'num_beams': 1, 'do_sample': False, 'max_new_tokens': 256}


## 3. Resolve project paths


In [4]:
def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, cwd.parent]:
        if (candidate / "data").exists() and (candidate / "runs").exists():
            return candidate
    return cwd


PROJECT_ROOT = find_project_root()

prediction_candidates = [
    PROJECT_ROOT / "runs/direct_full_run_1788151795/direct_predictions.csv",
    PROJECT_ROOT / "runs/direct_run_1788151795/direct_predictions.csv",
]

DIRECT_PREDICTIONS_PATH = next(
    (p for p in prediction_candidates if p.is_file()),
    prediction_candidates[0],
)

AUDIO_DIR = PROJECT_ROOT / "data/final_sample/final_sample_audio"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "runs"
    / f"direct_literal_unk_blocking_{int(time.time())}"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

if not DIRECT_PREDICTIONS_PATH.is_file():
    raise FileNotFoundError(f"Direct predictions not found: {DIRECT_PREDICTIONS_PATH}")

if not AUDIO_DIR.is_dir():
    raise NotADirectoryError(f"Audio directory not found: {AUDIO_DIR}")

print("Project root:", PROJECT_ROOT)
print("Official predictions:", DIRECT_PREDICTIONS_PATH)
print("Audio:", AUDIO_DIR)
print("Output:", OUTPUT_DIR)


Project root: /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research
Official predictions: /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/runs/direct_full_run_1788151795/direct_predictions.csv
Audio: /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/data/final_sample/final_sample_audio
Output: /Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/runs/direct_literal_unk_blocking_1789970452


## 4. Load official Direct predictions and select affected clips


In [5]:
pred = pd.read_csv(DIRECT_PREDICTIONS_PATH)

required = {
    "id",
    "clip",
    "sentence",
    "primary_accent",
    "sample_id",
    "reference_translation_zh",
    "direct_translation",
}

missing = required - set(pred.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")

pred = pred.copy()

pred["original_has_literal_unk"] = (
    pred["direct_translation"]
    .fillna("")
    .astype(str)
    .str.contains("<unk>", regex=False)
)

affected = pred[pred["original_has_literal_unk"]].copy()

print("Official rows:", len(pred))
print("Originally affected rows:", len(affected))

if TEST_SCOPE == "balanced_small":
    parts = []
    for accent, group in affected.groupby("primary_accent", sort=True):
        n = min(SAMPLES_PER_ACCENT, len(group))
        parts.append(group.sample(n=n, random_state=SEED))
    test_df = (
        pd.concat(parts, ignore_index=True)
        .sort_values(["primary_accent", "id"])
        .reset_index(drop=True)
    )
elif TEST_SCOPE == "all_affected":
    test_df = affected.reset_index(drop=True)
else:
    raise ValueError("TEST_SCOPE must be 'balanced_small' or 'all_affected'.")

print("Selected rows:", len(test_df))
display(test_df["primary_accent"].value_counts().rename("clips").to_frame())
display(
    test_df[
        ["id", "primary_accent", "sentence", "reference_translation_zh", "direct_translation"]
    ].head(20)
)


Official rows: 4200
Originally affected rows: 508
Selected rows: 35


,clips
primary_accent,
England English,5
Filipino,5
Hong Kong English,5
"India and South Asia (India, Pakistan, Sri Lanka)",5
Malaysian English,5
"Southern African (South Africa, Zimbabwe, Namibia)",5
United States English,5


,id,primary_accent,sentence,reference_translation_zh,direct_translation
0,sample_0090,England English,Restoration silver is characterized by embosse...,修复银的特点是压花图案表现的郁金香和自然的水果和树叶。,修复银的特点是<unk>花和自然的水果和叶子的雕刻图案.
1,sample_0221,England English,Who is the blonde girl with the red skirt?,那个穿红裙子的金发女孩是谁?,那个穿着红色<unk>子的金发女孩是谁?
2,sample_0235,England English,"When he saw Henderson in his garden, he called...",当他看见 Henderson 在他的花园里时，他隔着篱笆喊了一声，对方明白了他的意思。,"当他看到亨德森在他的花园,他叫过<unk>,并让自己理解."
3,sample_0282,England English,The hail pattered on the burnt brown grass.,冰雹拍打着烧焦的棕色草地。,冰雹在烧焦的棕色草地上<unk>.
4,sample_0441,England English,"Roll the dice, please.",请掷骰子。,"投<unk>子,请."
5,sample_0653,Filipino,"Grime, rust and residual particles polluted th...",尘垢、铁锈和残留的微粒污染了空气。,"污垢,<unk>和残留的粒子污染了空气."
6,sample_0731,Filipino,A little scrambling is required for access.,访问需要加一些扰码。,需要一点<unk>密才能进入.
7,sample_0826,Filipino,"The Englishman prodded him, and the boy asked ...",英国人戳了他一下，男孩就问她是谁治好了人们的病。,"这位英国人<unk>了他,男孩问她关于治愈人们疾病的人."
8,sample_0905,Filipino,"They walked in from the rain, all dishevelled ...",他们从雨中走了进来，衣冠不整，浑身湿透。,"他们从雨中走进来, 所有的<unk>乱和蒸汽."
9,sample_0965,Filipino,"You were looking squeamish this afternoon, he ...",他说着，你今天下午看起来很邋遢。,"他开始说,今天下午你看起来很<unk>."


## 5. Device and precision

CPU/FP32 can be used for debugging. T4/FP16 is preferred for the final sensitivity result.


In [6]:
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = torch.device("cuda" if CUDA_AVAILABLE else "cpu")
MODEL_DTYPE = torch.float16 if CUDA_AVAILABLE else torch.float32
GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else None

print("Device:", DEVICE)
print("dtype:", MODEL_DTYPE)
print("GPU:", GPU_NAME)

if not CUDA_AVAILABLE:
    print(
        "WARNING: CPU/FP32 differs from the original T4/FP16 run. "
        "Check baseline reproduction before interpreting the result."
    )


Device: cpu
dtype: torch.float32
GPU: None


## 6. Load the exact SeamlessM4T checkpoint


In [7]:
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = SeamlessM4Tv2ForSpeechToText.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=MODEL_DTYPE,
)

model.to(DEVICE)
model.eval()

MODEL_SAMPLE_RATE = int(
    getattr(processor.feature_extractor, "sampling_rate", 16000)
)

TOKENIZER = processor.tokenizer
SPECIAL_UNK_ID = TOKENIZER.unk_token_id

print("Model:", MODEL_ID)
print("Revision:", MODEL_REVISION)
print("Sample rate:", MODEL_SAMPLE_RATE)
print("Special UNK token:", TOKENIZER.unk_token)
print("Special UNK token ID:", SPECIAL_UNK_ID)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model: facebook/seamless-m4t-v2-large
Revision: 5f8cc790b19fc3f67a61c105133b20b34e3dcb76
Sample rate: 16000
Special UNK token: <unk>
Special UNK token ID: 1


## 7. Audio loader

The preferred path uses `torchaudio.load()`. If local MP3 decoding is unavailable, it falls back to `librosa`.


In [8]:
def load_audio_for_model(audio_path, target_sample_rate=MODEL_SAMPLE_RATE):
    audio_path = Path(audio_path)

    if not audio_path.is_file():
        raise FileNotFoundError(f"Audio file not found: {audio_path}")

    try:
        waveform, source_sample_rate = torchaudio.load(str(audio_path))

        if waveform.ndim != 2 or waveform.shape[1] == 0:
            raise ValueError(f"Invalid audio shape: {tuple(waveform.shape)}")

        waveform = waveform.mean(dim=0)

        if source_sample_rate != target_sample_rate:
            waveform = torchaudio.functional.resample(
                waveform,
                source_sample_rate,
                target_sample_rate,
            )

        waveform = waveform.to(torch.float32).contiguous()
        loader_used = "torchaudio"

    except RuntimeError:
        import librosa

        waveform_np, _ = librosa.load(
            str(audio_path),
            sr=target_sample_rate,
            mono=True,
        )

        waveform = torch.from_numpy(
            np.asarray(waveform_np)
        ).to(torch.float32)

        loader_used = "librosa_fallback"

    duration_sec = waveform.numel() / float(target_sample_rate)

    if duration_sec <= 0:
        raise ValueError(f"Decoded audio is empty: {audio_path}")

    return waveform.cpu().numpy(), duration_sec, loader_used


## 8. Generation helper

The constrained run differs only by adding `bad_words_ids`.


In [9]:
def synchronize_device():
    if CUDA_AVAILABLE:
        torch.cuda.synchronize()


def run_translation(audio_path, bad_words_ids=None):
    waveform, duration_sec, loader_used = load_audio_for_model(audio_path)

    inputs = processor(
        audio=waveform,
        sampling_rate=MODEL_SAMPLE_RATE,
        return_tensors="pt",
    )

    model_inputs = {}

    for name, tensor in inputs.items():
        if torch.is_floating_point(tensor):
            model_inputs[name] = tensor.to(
                device=DEVICE,
                dtype=MODEL_DTYPE,
            )
        else:
            model_inputs[name] = tensor.to(DEVICE)

    generation_kwargs = dict(GENERATION_KWARGS)

    if bad_words_ids:
        generation_kwargs["bad_words_ids"] = bad_words_ids

    synchronize_device()
    start = time.perf_counter()

    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            tgt_lang=TARGET_LANGUAGE,
            **generation_kwargs,
        )

    synchronize_device()
    runtime_sec = time.perf_counter() - start

    token_ids = generated[0]
    if token_ids.ndim > 1:
        token_ids = token_ids[0]

    token_ids_list = token_ids.detach().cpu().tolist()

    translation = processor.decode(
        token_ids_list,
        skip_special_tokens=True,
    ).strip()

    return {
        "translation": translation,
        "runtime_sec": runtime_sec,
        "audio_duration_sec": duration_sec,
        "loader_used": loader_used,
        "generated_token_count": len(token_ids_list),
        "generated_token_ids": token_ids_list,
        "contains_literal_unk": "<unk>" in translation,
        "special_unk_id_count": sum(
            int(token_id == SPECIAL_UNK_ID)
            for token_id in token_ids_list
        ),
    }


## 9. Warm-up


In [10]:
warmup_row = test_df.iloc[0]
warmup_audio = AUDIO_DIR / str(warmup_row["clip"])

_ = run_translation(
    warmup_audio,
    bad_words_ids=None,
)

print("Warm-up complete.")


/Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/.venv/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


Warm-up complete.


## 10. Re-run baseline on selected clips

Fresh token IDs are needed so the notebook can discover how the model actually spells literal `<unk>`.


In [11]:
baseline_records = []

for i, row in test_df.iterrows():
    audio_path = AUDIO_DIR / str(row["clip"])

    print(f"[{i + 1}/{len(test_df)}] Baseline | {row['id']}")

    result = run_translation(
        audio_path,
        bad_words_ids=None,
    )

    baseline_records.append({
        "id": row["id"],
        "sample_id": row["sample_id"],
        "primary_accent": row["primary_accent"],
        "clip": row["clip"],
        "sentence": row["sentence"],
        "reference_translation_zh": row["reference_translation_zh"],
        "official_direct_translation": row["direct_translation"],
        **result,
    })

baseline_df = pd.DataFrame(baseline_records)

baseline_df["exactly_matches_official"] = (
    baseline_df["translation"].fillna("").astype(str)
    == baseline_df["official_direct_translation"].fillna("").astype(str)
)

print(
    "Exact baseline reproduction:",
    f'{100 * baseline_df["exactly_matches_official"].mean():.2f}%'
)

print(
    "Fresh baseline outputs containing literal <unk>:",
    int(baseline_df["contains_literal_unk"].sum()),
    "/",
    len(baseline_df),
)

print(
    "Special UNK token-ID occurrences:",
    int(baseline_df["special_unk_id_count"].sum()),
)


[1/35] Baseline | sample_0090
[2/35] Baseline | sample_0221
[3/35] Baseline | sample_0235
[4/35] Baseline | sample_0282
[5/35] Baseline | sample_0441
[6/35] Baseline | sample_0653
[7/35] Baseline | sample_0731
[8/35] Baseline | sample_0826
[9/35] Baseline | sample_0905
[10/35] Baseline | sample_0965
[11/35] Baseline | sample_1253
[12/35] Baseline | sample_1265
[13/35] Baseline | sample_1383
[14/35] Baseline | sample_1555
[15/35] Baseline | sample_1731
[16/35] Baseline | sample_1864
[17/35] Baseline | sample_2041
[18/35] Baseline | sample_2061
[19/35] Baseline | sample_2093
[20/35] Baseline | sample_2176
[21/35] Baseline | sample_2442
[22/35] Baseline | sample_2724
[23/35] Baseline | sample_2757
[24/35] Baseline | sample_2779
[25/35] Baseline | sample_2971
[26/35] Baseline | sample_3082
[27/35] Baseline | sample_3131
[28/35] Baseline | sample_3220
[29/35] Baseline | sample_3340
[30/35] Baseline | sample_3536
[31/35] Baseline | sample_3699
[32/35] Baseline | sample_3744
[33/35] Baseline 

## 11. Discover the ordinary-token sequences that decode to literal `<unk>`

The notebook searches contiguous generated token n-grams and keeps sequences for which:

```python
tokenizer.decode(sequence, skip_special_tokens=False).strip() == "<unk>"
```


In [12]:
def discover_literal_unk_sequences(
    generated_token_lists,
    tokenizer,
    max_ngram=6,
):
    candidates = {}
    special_ids = set(tokenizer.all_special_ids)

    for token_ids in generated_token_lists:
        n_tokens = len(token_ids)

        for start in range(n_tokens):
            for length in range(1, max_ngram + 1):
                end = start + length

                if end > n_tokens:
                    break

                seq = tuple(token_ids[start:end])

                if any(token_id in special_ids for token_id in seq):
                    continue

                decoded = tokenizer.decode(
                    list(seq),
                    skip_special_tokens=False,
                ).strip()

                if decoded == "<unk>":
                    candidates[seq] = candidates.get(seq, 0) + 1

    # Keep shortest useful sequences first.
    ordered = sorted(
        candidates.items(),
        key=lambda x: (len(x[0]), -x[1], x[0]),
    )

    kept = []

    for seq, count in ordered:
        contains_shorter = False

        for shorter, _ in kept:
            if len(shorter) >= len(seq):
                continue

            for i in range(len(seq) - len(shorter) + 1):
                if tuple(seq[i:i + len(shorter)]) == tuple(shorter):
                    contains_shorter = True
                    break

            if contains_shorter:
                break

        if not contains_shorter:
            kept.append((seq, count))

    return kept


discovered = discover_literal_unk_sequences(
    baseline_df["generated_token_ids"].tolist(),
    TOKENIZER,
    max_ngram=MAX_LITERAL_UNK_NGRAM,
)

if not discovered:
    raise RuntimeError(
        "No ordinary-token sequence decoding exactly to '<unk>' was discovered."
    )

rows = []

for seq, count in discovered:
    rows.append({
        "token_ids": list(seq),
        "tokens": TOKENIZER.convert_ids_to_tokens(list(seq)),
        "decoded": TOKENIZER.decode(
            list(seq),
            skip_special_tokens=False,
        ),
        "observed_ngram_occurrences": count,
    })

literal_sequence_df = pd.DataFrame(rows)
display(literal_sequence_df)

LITERAL_UNK_SEQUENCES = [
    item["token_ids"]
    for item in rows
]

print("Sequences that will be blocked:")
print(json.dumps(LITERAL_UNK_SEQUENCES, indent=2))


,token_ids,tokens,decoded,observed_ngram_occurrences
0,"[249371, 2105, 248948]","[<, unk, >]",<unk>,37
1,"[9614, 2105, 248948]","[▁<, unk, >]",<unk>,2


Sequences that will be blocked:
[
  [
    249371,
    2105,
    248948
  ],
  [
    9614,
    2105,
    248948
  ]
]


## 12. Sanity-check coverage of the discovered sequences


In [13]:
def count_subsequence(sequence, pattern):
    if not pattern or len(pattern) > len(sequence):
        return 0

    count = 0

    for i in range(len(sequence) - len(pattern) + 1):
        if sequence[i:i + len(pattern)] == pattern:
            count += 1

    return count


coverage_rows = []

for _, row in baseline_df.iterrows():
    token_ids = row["generated_token_ids"]

    matches = sum(
        count_subsequence(token_ids, pattern)
        for pattern in LITERAL_UNK_SEQUENCES
    )

    coverage_rows.append({
        "id": row["id"],
        "contains_literal_unk": row["contains_literal_unk"],
        "matching_blocked_sequences": matches,
    })

coverage_df = pd.DataFrame(coverage_rows)

print(
    "Literal-<unk> baseline clips with discovered sequence coverage:",
    int(
        (
            coverage_df["contains_literal_unk"]
            & coverage_df["matching_blocked_sequences"].gt(0)
        ).sum()
    ),
    "/",
    int(coverage_df["contains_literal_unk"].sum()),
)

display(
    coverage_df[
        coverage_df["contains_literal_unk"]
    ].head(30)
)


Literal-<unk> baseline clips with discovered sequence coverage: 35 / 35


,id,contains_literal_unk,matching_blocked_sequences
0,sample_0090,True,1
1,sample_0221,True,1
2,sample_0235,True,1
3,sample_0282,True,1
4,sample_0441,True,1
5,sample_0653,True,1
6,sample_0731,True,1
7,sample_0826,True,1
8,sample_0905,True,1
9,sample_0965,True,1


## 13. Run constrained decoding

The only intended change is:

```python
bad_words_ids=LITERAL_UNK_SEQUENCES
```


In [14]:
blocked_records = []

for i, row in test_df.iterrows():
    audio_path = AUDIO_DIR / str(row["clip"])

    print(f"[{i + 1}/{len(test_df)}] Block literal <unk> | {row['id']}")

    result = run_translation(
        audio_path,
        bad_words_ids=LITERAL_UNK_SEQUENCES,
    )

    blocked_records.append({
        "id": row["id"],
        "sample_id": row["sample_id"],
        "primary_accent": row["primary_accent"],
        "clip": row["clip"],
        "sentence": row["sentence"],
        "reference_translation_zh": row["reference_translation_zh"],
        "official_direct_translation": row["direct_translation"],
        **result,
    })

blocked_df = pd.DataFrame(blocked_records)

print(
    "Blocked outputs still containing literal <unk>:",
    int(blocked_df["contains_literal_unk"].sum()),
    "/",
    len(blocked_df),
)


[1/35] Block literal <unk> | sample_0090


/Users/nlh/Downloads/uoa_lectures/CS760/project/cs760-accent-st-robustness-ml-research/.venv/lib/python3.9/site-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(


[2/35] Block literal <unk> | sample_0221
[3/35] Block literal <unk> | sample_0235
[4/35] Block literal <unk> | sample_0282
[5/35] Block literal <unk> | sample_0441
[6/35] Block literal <unk> | sample_0653
[7/35] Block literal <unk> | sample_0731
[8/35] Block literal <unk> | sample_0826
[9/35] Block literal <unk> | sample_0905
[10/35] Block literal <unk> | sample_0965
[11/35] Block literal <unk> | sample_1253
[12/35] Block literal <unk> | sample_1265
[13/35] Block literal <unk> | sample_1383
[14/35] Block literal <unk> | sample_1555
[15/35] Block literal <unk> | sample_1731
[16/35] Block literal <unk> | sample_1864
[17/35] Block literal <unk> | sample_2041
[18/35] Block literal <unk> | sample_2061
[19/35] Block literal <unk> | sample_2093
[20/35] Block literal <unk> | sample_2176
[21/35] Block literal <unk> | sample_2442
[22/35] Block literal <unk> | sample_2724
[23/35] Block literal <unk> | sample_2757
[24/35] Block literal <unk> | sample_2779
[25/35] Block literal <unk> | sample_2971


## 14. Compare baseline and blocked outputs


In [15]:
comparison = baseline_df[
    [
        "id",
        "primary_accent",
        "sentence",
        "reference_translation_zh",
        "official_direct_translation",
        "translation",
        "contains_literal_unk",
        "runtime_sec",
    ]
].rename(
    columns={
        "translation": "baseline_translation",
        "contains_literal_unk": "baseline_has_literal_unk",
        "runtime_sec": "baseline_runtime_sec",
    }
)

comparison = comparison.merge(
    blocked_df[
        [
            "id",
            "translation",
            "contains_literal_unk",
            "runtime_sec",
        ]
    ].rename(
        columns={
            "translation": "blocked_translation",
            "contains_literal_unk": "blocked_has_literal_unk",
            "runtime_sec": "blocked_runtime_sec",
        }
    ),
    on="id",
    validate="one_to_one",
)

comparison["translation_changed"] = (
    comparison["baseline_translation"]
    != comparison["blocked_translation"]
)

comparison["literal_unk_removed"] = (
    comparison["baseline_has_literal_unk"]
    & ~comparison["blocked_has_literal_unk"]
)

summary = pd.DataFrame([
    {
        "condition": "baseline_greedy",
        "clips": len(comparison),
        "literal_unk_outputs": int(
            comparison["baseline_has_literal_unk"].sum()
        ),
        "mean_runtime_sec": comparison["baseline_runtime_sec"].mean(),
    },
    {
        "condition": "greedy_block_literal_unk",
        "clips": len(comparison),
        "literal_unk_outputs": int(
            comparison["blocked_has_literal_unk"].sum()
        ),
        "mean_runtime_sec": comparison["blocked_runtime_sec"].mean(),
    },
])

summary["literal_unk_rate_pct"] = (
    100 * summary["literal_unk_outputs"] / summary["clips"]
)

display(summary)

print(
    "Translations changed:",
    int(comparison["translation_changed"].sum()),
    "/",
    len(comparison),
)

print(
    "Literal <unk> removed:",
    int(comparison["literal_unk_removed"].sum()),
    "/",
    int(comparison["baseline_has_literal_unk"].sum()),
)

display(
    comparison[
        [
            "id",
            "primary_accent",
            "sentence",
            "reference_translation_zh",
            "baseline_translation",
            "blocked_translation",
            "literal_unk_removed",
        ]
    ].head(50)
)


,condition,clips,literal_unk_outputs,mean_runtime_sec,literal_unk_rate_pct
0,baseline_greedy,35,35,2.023621,100.0
1,greedy_block_literal_unk,35,0,3.002300,0.0


Translations changed: 35 / 35
Literal <unk> removed: 35 / 35


,id,primary_accent,sentence,reference_translation_zh,baseline_translation,blocked_translation,literal_unk_removed
0,sample_0090,England English,Restoration silver is characterized by embosse...,修复银的特点是压花图案表现的郁金香和自然的水果和树叶。,修复银的特点是<unk>花和自然的水果和叶子的雕刻图案.,修复银的特点是<unk > 郁金香和自然的水果和叶子的雕刻图案.,True
1,sample_0221,England English,Who is the blonde girl with the red skirt?,那个穿红裙子的金发女孩是谁?,那个穿着红色<unk>子的金发女孩是谁?,那个穿着红色<unk > 裙子的金发女孩是谁?,True
2,sample_0235,England English,"When he saw Henderson in his garden, he called...",当他看见 Henderson 在他的花园里时，他隔着篱笆喊了一声，对方明白了他的意思。,"当他看到亨德森在他的花园,他叫过<unk>,并让自己理解.","当他看到亨德森在他的花园,他叫过<unk > 围<unk > 和让自己理解.",True
3,sample_0282,England English,The hail pattered on the burnt brown grass.,冰雹拍打着烧焦的棕色草地。,冰雹在烧焦的棕色草地上<unk>.,冰雹在烧焦的棕色草地上<unk >.,True
4,sample_0441,England English,"Roll the dice, please.",请掷骰子。,"投<unk>子,请.","投<unk >,请.",True
5,sample_0653,Filipino,"Grime, rust and residual particles polluted th...",尘垢、铁锈和残留的微粒污染了空气。,"污垢,<unk>和残留的粒子污染了空气.","污垢,<unk >和残留的粒子污染了空气.",True
6,sample_0731,Filipino,A little scrambling is required for access.,访问需要加一些扰码。,需要一点<unk>密才能进入.,需要一点<unk > 浏览才能访问.,True
7,sample_0826,Filipino,"The Englishman prodded him, and the boy asked ...",英国人戳了他一下，男孩就问她是谁治好了人们的病。,"这位英国人<unk>了他,男孩问她关于治愈人们疾病的人.","这位英国人<unk > 了他,男孩问她关于治愈人们疾病的人.",True
8,sample_0905,Filipino,"They walked in from the rain, all dishevelled ...",他们从雨中走了进来，衣冠不整，浑身湿透。,"他们从雨中走进来, 所有的<unk>乱和蒸汽.","他们从雨中走进来, 所有的<unk > 乱和蒸汽.",True
9,sample_0965,Filipino,"You were looking squeamish this afternoon, he ...",他说着，你今天下午看起来很邋遢。,"他开始说,你今天下午看起来很<unk>.","他开始说,你今天下午看起来很<unk >.",True


## 15. BLEU, chrF, and chrF++ on the same clips

This determines whether blocking `<unk>` improves translation quality or merely hides the marker.


In [16]:
from sacrebleu.metrics import BLEU, CHRF

bleu_metric = BLEU(tokenize="zh")
chrf_metric = CHRF(beta=2, word_order=0)
chrfpp_metric = CHRF(beta=2, word_order=2)


def corpus_metrics(hypotheses, references):
    return {
        "BLEU_zh": bleu_metric.corpus_score(
            hypotheses,
            [references],
        ).score,
        "chrF": chrf_metric.corpus_score(
            hypotheses,
            [references],
        ).score,
        "chrFpp": chrfpp_metric.corpus_score(
            hypotheses,
            [references],
        ).score,
    }


references = comparison["reference_translation_zh"].astype(str).tolist()

baseline_metrics = corpus_metrics(
    comparison["baseline_translation"].astype(str).tolist(),
    references,
)

blocked_metrics = corpus_metrics(
    comparison["blocked_translation"].astype(str).tolist(),
    references,
)

metric_summary = pd.DataFrame([
    {
        "condition": "baseline_greedy",
        **baseline_metrics,
    },
    {
        "condition": "greedy_block_literal_unk",
        **blocked_metrics,
    },
])

display(metric_summary)

print("BLEU signature:", bleu_metric.get_signature())
print("chrF signature:", chrf_metric.get_signature())
print("chrF++ signature:", chrfpp_metric.get_signature())


,condition,BLEU_zh,chrF,chrFpp
0,baseline_greedy,28.186585,24.850456,18.863623
1,greedy_block_literal_unk,15.515017,19.147499,14.487824


BLEU signature: nrefs:1|case:mixed|eff:no|tok:zh|smooth:exp|version:2.6.0
chrF signature: nrefs:1|case:mixed|eff:yes|nc:6|nw:0|space:no|version:2.6.0
chrF++ signature: nrefs:1|case:mixed|eff:yes|nc:6|nw:2|space:no|version:2.6.0


## 16. Manual review

A result is not automatically better just because `<unk>` disappears. Inspect the replacement text.


In [17]:
manual_review = comparison[
    comparison["translation_changed"]
][
    [
        "id",
        "primary_accent",
        "sentence",
        "reference_translation_zh",
        "baseline_translation",
        "blocked_translation",
        "literal_unk_removed",
    ]
].copy()

display(manual_review)

manual_review.to_csv(
    OUTPUT_DIR / "manual_review_changed_outputs.csv",
    index=False,
    encoding="utf-8-sig",
)


,id,primary_accent,sentence,reference_translation_zh,baseline_translation,blocked_translation,literal_unk_removed
0,sample_0090,England English,Restoration silver is characterized by embosse...,修复银的特点是压花图案表现的郁金香和自然的水果和树叶。,修复银的特点是<unk>花和自然的水果和叶子的雕刻图案.,修复银的特点是<unk > 郁金香和自然的水果和叶子的雕刻图案.,True
1,sample_0221,England English,Who is the blonde girl with the red skirt?,那个穿红裙子的金发女孩是谁?,那个穿着红色<unk>子的金发女孩是谁?,那个穿着红色<unk > 裙子的金发女孩是谁?,True
2,sample_0235,England English,"When he saw Henderson in his garden, he called...",当他看见 Henderson 在他的花园里时，他隔着篱笆喊了一声，对方明白了他的意思。,"当他看到亨德森在他的花园,他叫过<unk>,并让自己理解.","当他看到亨德森在他的花园,他叫过<unk > 围<unk > 和让自己理解.",True
3,sample_0282,England English,The hail pattered on the burnt brown grass.,冰雹拍打着烧焦的棕色草地。,冰雹在烧焦的棕色草地上<unk>.,冰雹在烧焦的棕色草地上<unk >.,True
4,sample_0441,England English,"Roll the dice, please.",请掷骰子。,"投<unk>子,请.","投<unk >,请.",True
5,sample_0653,Filipino,"Grime, rust and residual particles polluted th...",尘垢、铁锈和残留的微粒污染了空气。,"污垢,<unk>和残留的粒子污染了空气.","污垢,<unk >和残留的粒子污染了空气.",True
6,sample_0731,Filipino,A little scrambling is required for access.,访问需要加一些扰码。,需要一点<unk>密才能进入.,需要一点<unk > 浏览才能访问.,True
7,sample_0826,Filipino,"The Englishman prodded him, and the boy asked ...",英国人戳了他一下，男孩就问她是谁治好了人们的病。,"这位英国人<unk>了他,男孩问她关于治愈人们疾病的人.","这位英国人<unk > 了他,男孩问她关于治愈人们疾病的人.",True
8,sample_0905,Filipino,"They walked in from the rain, all dishevelled ...",他们从雨中走了进来，衣冠不整，浑身湿透。,"他们从雨中走进来, 所有的<unk>乱和蒸汽.","他们从雨中走进来, 所有的<unk > 乱和蒸汽.",True
9,sample_0965,Filipino,"You were looking squeamish this afternoon, he ...",他说着，你今天下午看起来很邋遢。,"他开始说,你今天下午看起来很<unk>.","他开始说,你今天下午看起来很<unk >.",True


## 17. Save outputs


In [18]:
baseline_df.to_csv(
    OUTPUT_DIR / "baseline_rerun.csv",
    index=False,
    encoding="utf-8-sig",
)

blocked_df.to_csv(
    OUTPUT_DIR / "literal_unk_blocked_predictions.csv",
    index=False,
    encoding="utf-8-sig",
)

comparison.to_csv(
    OUTPUT_DIR / "side_by_side_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

literal_sequence_df.to_csv(
    OUTPUT_DIR / "discovered_literal_unk_sequences.csv",
    index=False,
)

summary.to_csv(
    OUTPUT_DIR / "unk_rate_comparison.csv",
    index=False,
)

metric_summary.to_csv(
    OUTPUT_DIR / "translation_metric_comparison.csv",
    index=False,
)

run_summary = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "purpose": "Literal-<unk> constrained-decoding sensitivity analysis",
    "test_scope": TEST_SCOPE,
    "selected_clips": int(len(test_df)),
    "original_affected_rows_total": int(len(affected)),
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "target_language": TARGET_LANGUAGE,
    "device": str(DEVICE),
    "dtype": str(MODEL_DTYPE),
    "gpu": GPU_NAME,
    "pytorch_version": torch.__version__,
    "torchaudio_version": torchaudio.__version__,
    "transformers_version": transformers.__version__,
    "sacrebleu_version": sacrebleu.__version__,
    "generation_settings": GENERATION_KWARGS,
    "special_unk_token": TOKENIZER.unk_token,
    "special_unk_token_id": SPECIAL_UNK_ID,
    "literal_unk_sequences_blocked": LITERAL_UNK_SEQUENCES,
    "baseline_exact_reproduction_rate": float(
        baseline_df["exactly_matches_official"].mean()
    ),
    "baseline_literal_unk_outputs": int(
        comparison["baseline_has_literal_unk"].sum()
    ),
    "blocked_literal_unk_outputs": int(
        comparison["blocked_has_literal_unk"].sum()
    ),
    "literal_unk_removed_count": int(
        comparison["literal_unk_removed"].sum()
    ),
    "translations_changed": int(
        comparison["translation_changed"].sum()
    ),
    "baseline_metrics": baseline_metrics,
    "blocked_metrics": blocked_metrics,
    "note": (
        "Sensitivity analysis only. Official 4,200-sample Direct "
        "predictions remain unchanged."
    ),
}

(OUTPUT_DIR / "run_summary.json").write_text(
    json.dumps(run_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print(json.dumps(run_summary, indent=2, ensure_ascii=False))
print("\nSaved to:", OUTPUT_DIR)


{
  "created_utc": "2026-09-21T06:04:12.163196+00:00",
  "purpose": "Literal-<unk> constrained-decoding sensitivity analysis",
  "test_scope": "balanced_small",
  "selected_clips": 35,
  "original_affected_rows_total": 508,
  "model_id": "facebook/seamless-m4t-v2-large",
  "model_revision": "5f8cc790b19fc3f67a61c105133b20b34e3dcb76",
  "target_language": "cmn",
  "device": "cpu",
  "dtype": "torch.float32",
  "gpu": null,
  "pytorch_version": "2.8.0",
  "torchaudio_version": "2.8.0",
  "transformers_version": "4.57.6",
  "sacrebleu_version": "2.6.0",
  "generation_settings": {
    "num_beams": 1,
    "do_sample": false,
    "max_new_tokens": 256
  },
  "special_unk_token": "<unk>",
  "special_unk_token_id": 1,
  "literal_unk_sequences_blocked": [
    [
      249371,
      2105,
      248948
    ],
    [
      9614,
      2105,
      248948
    ]
  ],
  "baseline_exact_reproduction_rate": 0.9428571428571428,
  "baseline_literal_unk_outputs": 35,
  "blocked_literal_unk_outputs": 0,
  "li

## How to judge whether it worked

A **strong positive result** means:

- literal `<unk>` disappears from most/all affected outputs;
- BLEU/chrF/chrF++ improve on the same clips;
- manual inspection shows sensible Chinese replacements.

A **cosmetic-only result** means `<unk>` disappears but the new words are still wrong.

If literal `<unk>` still appears, the model found another token sequence that spells it, so the discovery/blocking step needs to be expanded.
